# Notebook 1: Data Cleaning & Exploratory Data Analysis
## Federal Reserve Interest Rate Prediction — ML Pipeline

**Objective:** Load, clean, and explore the FRED economic dataset covering 1954–2024.

**Dataset Features:**
- `FEDRates` — Federal Funds Rate (TARGET)
- `ConsumerPriceIndexAllItems` — CPI % change (MoM)
- `GDP` — Nominal GDP (Billions USD)
- `InflationConsumerPrice` — Annual inflation rate
- `MedianConsumerPriceIndex` — Median CPI
- `RealGDP` — Inflation-adjusted GDP
- `RealGDPPerCapita` — Real GDP per person
- `RealPotentialGDP` — Economy's productive capacity
- `UnemployemenrRate` — U-3 unemployment rate


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"


## 1. Data Loading

In [ ]:
# Load dataset
df_raw = pd.read_csv(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values('date').reset_index(drop=True)

print(f"Shape: {df_raw.shape}")
print(f"Date range: {df_raw['date'].min().date()} to {df_raw['date'].max().date()}")
display(df_raw.head(10))


In [ ]:
# Descriptive statistics
display(df_raw.describe().T.round(3))


In [ ]:
# Missing value analysis
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
display(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}))


## 2. Data Cleaning

In [ ]:
df = df_raw.copy()

# Step 1: Drop rows with missing target
before = len(df)
df = df.dropna(subset=['FEDRates'])
print(f"Rows dropped (missing FEDRates): {before - len(df)}")

# Step 2: Forward-fill then backward-fill remaining NaNs
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().bfill()
print(f"Remaining NaNs after fill: {df[numeric_cols].isnull().sum().sum()}")

# Step 3: Fix zero CPI entries (sentinel value for unavailable data)
zero_mask = df['ConsumerPriceIndexAllItems'] == 0.0
print(f"Zero CPI entries: {zero_mask.sum()}")
df.loc[zero_mask, 'ConsumerPriceIndexAllItems'] = np.nan
df['ConsumerPriceIndexAllItems'] = df['ConsumerPriceIndexAllItems'].interpolate('linear').ffill().bfill()

# Step 4: Winsorization — cap at 1st/99th percentile
for col in numeric_cols:
    p01, p99 = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lower=p01, upper=p99)

print(f"Final shape: {df.shape}")


In [ ]:
# Outlier detection using IQR method
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
features = ['ConsumerPriceIndexAllItems','GDP','InflationConsumerPrice',
            'MedianConsumerPriceIndex','RealGDP','RealGDPPerCapita',
            'RealPotentialGDP','UnemployemenrRate']
for i, col in enumerate(features):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    axes[i].boxplot([df_raw[col].dropna(), df[col]], labels=['Before','After'],
                    patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[i], alpha=0.6))
    axes[i].set_title(f'{col[:15]}\n({outliers} outliers)', fontsize=9, fontweight='bold')
fig.suptitle('Box Plots: Before vs After Winsorization', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Exploratory Data Analysis

In [ ]:
# FED Rate time series with recession periods
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['date'], df['FEDRates'], color='#2196F3', linewidth=1.5)
ax.fill_between(df['date'], df['FEDRates'], alpha=0.1, color='#2196F3')
recessions = [('1973-11','1975-03'),('1980-01','1980-07'),('1981-07','1982-11'),
              ('1990-07','1991-03'),('2001-03','2001-11'),('2007-12','2009-06'),
              ('2020-02','2020-04')]
for r in recessions:
    ax.axvspan(pd.to_datetime(r[0]), pd.to_datetime(r[1]), alpha=0.15, color='red')
ax.set_title('US Federal Funds Rate (1954–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Rate (%)')
plt.tight_layout(); plt.show()


In [ ]:
# Distribution plots for all features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
all_cols = ['FEDRates'] + features
for i, col in enumerate(all_cols):
    axes[i].hist(df[col], bins=40, color=PALETTE[i % len(PALETTE)], alpha=0.75, edgecolor='white')
    axes[i].axvline(df[col].mean(),   color='red',   linestyle='--', label='Mean')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label='Median')
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=7)
fig.suptitle('Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(11, 9))
corr = df[all_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Pearson Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("\nKey correlations with FEDRates:")
print(corr['FEDRates'].drop('FEDRates').sort_values(key=abs, ascending=False).round(4))


In [ ]:
# Scatter plots vs FED Rate
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
from scipy.stats import pearsonr
for i, col in enumerate(features):
    sc = axes[i].scatter(df[col], df['FEDRates'],
                         c=df['date'].astype(np.int64), cmap='viridis', alpha=0.4, s=15)
    r, p = pearsonr(df[col], df['FEDRates'])
    axes[i].set_title(f"{col[:14]}\nr={r:.3f}, p={p:.2e}", fontsize=9)
    axes[i].set_xlabel(col[:12]); axes[i].set_ylabel('FEDRates')
fig.suptitle('Feature vs FED Rate Scatter (Color = Time)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Summary
- 846 rows, 9 features loaded from FRED
- 111 zero CPI values fixed via interpolation
- Winsorization applied to cap extreme outliers
- Inflation shows the strongest positive correlation with FED Rates (r≈0.72)